# Klique LLM Gateway: Simple Setup

Get a first model response through the Klique LLM Gateway. The Gateway gives applications one governed entry point to models; **Simple Setup** creates a ready-made Profile, Routed Resource, and provider Resources in the UI.

This short walkthrough uses the **Cost first** setup: ordinary text requests go to Claude Haiku, while requests using tools go to Claude Sonnet. It makes one short text request so you can confirm the path works.

> **Provider note:** Simple Setup currently creates Anthropic Resources and asks for an Anthropic API key when it needs to create them. Anthropic is not required for the Gateway generally; administrators can configure other supported service providers as Resources separately.

> **Usage note:** Running the final request sends a prompt to your configured provider and may incur usage charges or count against a Klique budget. This notebook does not send a request until you run that cell.

**VIDEO CAPTURE: Show the Klique LLM Gateway page and introduce Simple Setup and the Cost first preset.**

## 1. Prerequisites

- Klique **1.3 or later**, where the current release notes introduce the one-step Gateway routing presets.
- Access to a deployed Klique LLM Gateway. For self-hosted Klique, the administrator must deploy the AI Application Gateway; hosted workspaces must have Gateway access enabled.
- An administrator or other account with permission to manage **Control → AI Resources → LLM Gateway**. If you do not have this access, ask an administrator to create the setup and grant you access to its Profile.
- An Anthropic API key for Simple Setup, unless the required Anthropic Resources already exist and can be reused. The Gateway stores and uses this provider key; it does not belong in this notebook.
- Python 3 and Jupyter. The request below uses only Python's standard library; no additional package is required.

The request needs a Gateway URL, a Profile name, a Profile model alias, and a Klique Gateway access token. The code uses the existing Klique notebook interface: `POST <gateway>/v1/chat/completions`, a Bearer token, and the `X-ClearML-Profile` header.

**VIDEO CAPTURE: Briefly show the prerequisites, then open Control → AI Resources → LLM Gateway.**

## 2. Create the Simple Setup

In **Control → AI Resources → LLM Gateway**, click **Simple Setup** beside **Resources, routing & profiles**. Choose **Cost first**. When prompted, enter the Anthropic API key for the Resources being created, then apply the setup. If all required Resources already exist, the UI says a key is not needed.

The setup creates a Profile, a Routed Resource, and any Anthropic Resources it needs. The Profile is open to everyone by default; review its access settings if your workspace needs narrower access. Setups reuse same-named Resources, including their existing credentials.

After applying the setup, open **My Setup → Profile → Your AI Endpoints** and note the Profile name and the alias for the new route. The setup exposes the Routed Resource as `auto` by default. Use the exact values displayed for your account.

> **VIDEO CAPTURE: Click Simple Setup, choose Cost first, show the Anthropic key field without exposing the key, apply the setup, then show the resulting Profile and `auto` alias in Your AI Endpoints.**

**Screenshot placeholder:** Simple Setup preset picker and the resulting Profile / `auto` alias. Do not capture or include the API key.

## 3. Set the request details securely

Fill in the Gateway base URL and Profile name shown for your workspace. `KLIQUE_GATEWAY_URL` should be the Gateway base ending in `/llm`, not the chat-completions path. Keep the alias set to `auto` if that is what Your AI Endpoints displays.

Load the Gateway token from an environment variable named `KLIQUE_GATEWAY_TOKEN` before starting Jupyter. Never paste a provider key or Gateway token into a notebook cell, commit it, or print it. The cell checks only whether the variable is present.

In [ ]:
import json
import os
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

GATEWAY_URL = "https://YOUR-KLIQUE-HOST/llm"  # Replace with the Gateway base URL
PROFILE = "YOUR-PROFILE-NAME"             # Copy the Profile name from Your AI Endpoints
MODEL_ALIAS = "auto"                       # Use the alias shown for this Simple Setup
GATEWAY_TOKEN = os.environ.get("KLIQUE_GATEWAY_TOKEN")

if not GATEWAY_TOKEN:
    raise RuntimeError("Set KLIQUE_GATEWAY_TOKEN in the environment before starting Jupyter.")
if not GATEWAY_URL.startswith("https://") or "YOUR-KLIQUE-HOST" in GATEWAY_URL:
    raise ValueError("Set GATEWAY_URL to your HTTPS Gateway base ending in /llm.")
if not PROFILE or PROFILE == "YOUR-PROFILE-NAME":
    raise ValueError("Set PROFILE to the exact name shown in Your AI Endpoints.")

GATEWAY_URL = GATEWAY_URL.rstrip("/")
print("Gateway settings are present. Token value was not displayed.")

## 4. Send one test request

This sends one OpenAI-compatible chat-completions request through the Gateway. A successful result prints the assistant's answer. HTTP errors report the status and a safe, short Gateway message without printing credentials.

**VIDEO CAPTURE: Run the request cell and show the answer returned through the Gateway.**

In [ ]:
request_body = {
    "model": MODEL_ALIAS,
    "messages": [{"role": "user", "content": "Explain the purpose of an LLM Gateway in one sentence."}],
    "max_tokens": 128,
}
request = Request(
    f"{GATEWAY_URL}/v1/chat/completions",
    data=json.dumps(request_body).encode("utf-8"),
    headers={
        "Authorization": f"Bearer {GATEWAY_TOKEN}",
        "X-ClearML-Profile": PROFILE,
        "Content-Type": "application/json",
    },
    method="POST",
)

try:
    with urlopen(request, timeout=60) as response:
        result = json.loads(response.read().decode("utf-8"))
    answer = result["choices"][0]["message"]["content"]
    if not answer:
        raise ValueError("The Gateway returned an empty assistant message.")
    print("Gateway request succeeded:\n")
    print(answer)
except HTTPError as error:
    detail = ""
    try:
        payload = json.loads(error.read(4096).decode("utf-8"))
        detail = payload.get("error", {}).get("message", "")
    except (UnicodeDecodeError, json.JSONDecodeError, AttributeError):
        pass
    print(f"Gateway returned HTTP {error.code}. {detail or 'Check the troubleshooting section.'}")
except URLError as error:
    print(f"Could not connect to the Gateway: {error.reason}. Check the URL, network, and TLS access.")
except (KeyError, IndexError, TypeError, json.JSONDecodeError) as error:
    print(f"The Gateway response did not contain the expected chat completion: {error}")

## 5. Verify the request

The printed response confirms that this test request completed. To confirm Gateway usage was recorded, open **Overview → AI Usage**, choose the Gateway source and a time range that includes the request, then inspect the usage breakdown. Gateway usage is metered automatically; the dashboard reports token use and usage details.

One successful call verifies basic connectivity, authentication, Profile access, alias selection, and an upstream response. It does not establish production readiness, capacity, availability, budget, or rate-limit behavior.

**VIDEO CAPTURE: Open Overview → AI Usage and show the new Gateway usage in the selected time range.**

**Screenshot placeholder:** AI Usage dashboard with Gateway usage for the demonstration request; avoid showing unrelated users' data.

## 6. Troubleshooting

| Symptom | Check |
|---|---|
| `401` or `403` | Check that `KLIQUE_GATEWAY_TOKEN` is a valid, unexpired Gateway token and that your identity may use the Profile. Do not paste the token into a support message. |
| `404` / `model_not_found` | Confirm the Profile name and exact model alias in **Your AI Endpoints**. The alias is not necessarily the upstream provider model ID. |
| `404` / route not found | Confirm the URL is the HTTPS Gateway base ending `/llm`; the code appends `/v1/chat/completions`. |
| Provider authentication or upstream error | Ask the Gateway administrator to check the Anthropic Resource credential and provider response. Simple Setup's key is configured in the Gateway UI, not in this notebook. |
| `429` | A Gateway Resource rate limit or spend quota may have been reached. Check with the administrator and review AI Usage / quota settings. |
| Connection error or timeout | Check VPN/network access, HTTPS/TLS, and the Gateway host. If the request may have reached the service, check AI Usage before retrying. |

The Gateway may return provider-specific error details; this notebook intentionally avoids assuming one status code for every upstream credential failure.

## 7. Next steps

Simple Setup is a starting point for a ready-made route. Move to manual Resources, Routed Resources, Profiles, and policies when you need a different provider, custom model endpoints, more specific access, or routing behavior beyond the presets. The existing `02a_Use_the_LLM_Gateway.ipynb` notebook covers making Gateway requests; `02b_Models_and_Routed_Resources.ipynb` explores multiple models and custom routing.

### Narration outline

1. Introduce the Gateway as one governed model entry point and explain that Simple Setup applies a ready-made route.
2. Show the Cost first preset and configure the Anthropic key in the UI without revealing it.
3. Find the generated Profile and `auto` alias in Your AI Endpoints.
4. Load the Gateway token from the environment, fill in the URL and Profile, and run the short test request.
5. Confirm the response and show its usage in AI Usage; close by pointing to the advanced Gateway notebook.

### References

- [LLM Gateway overview](https://klique.ai/docs/latest/docs/whats_klique/llm_gateway)
- [LLM Gateway settings and Simple Setup](https://klique.ai/docs/latest/docs/webapp/settings/webapp_settings_llm_gateway)
- [Klique 1.3 release notes](https://klique.ai/docs/latest/docs/whats_new/1.3)
- [AI Usage dashboard](https://klique.ai/docs/latest/docs/webapp/webapp_admin_dashboard_ai_usage)